# SIGNOVA — Continuous Sign Language Recognition Experiments

This notebook demonstrates the end-to-end Machine Learning pipeline for SIGNOVA:
1. **Environment & Hardware Inspection** (RTX 5060 Laptop GPU, CUDA 12.9, PyTorch 2.11)
2. **Vocabulary & Dataset Exploration** (ArabSign ArSL & How2Sign ASL)
3. **Model Inference & Benchmarking** using `load_model` and `predict`
4. **Direct Video-to-Text Pipeline** (`predict_video` with MediaPipe HandLandmarker)
5. **Error Analysis & Latency Profiling**

In [ ]:
import os
import sys
from pathlib import Path

# Fix Windows console encoding for Arabic characters
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

import torch
import numpy as np

# Robust repository root resolution (works from workspace root or notebooks folder)
def find_repo_root():
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "pyproject.toml").exists() and (p / "ml").exists():
            return p
    return cur

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Project Root: {repo_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 1. Load Trained ArabSign (ArSL) Model

In [ ]:
from ml.sign_language.src.inference import load_model, predict, predict_video

# Automatically loads model and vocab from artifacts/arabsign/
predictor = load_model(language="ar")
print(f"Loaded predictor for language: {predictor.language}")
print(f"Vocabulary size: {len(predictor.vocabulary)} tokens")

## 2. Real-time Inference on Continuous Keypoint Sequences

In [ ]:
# Simulate incoming continuous hand keypoints (60 frames, 126 coordinate features)
dummy_sequence = np.random.randn(60, 126).astype(np.float32)

result = predict(dummy_sequence, language="ar", predictor=predictor)
print("Prediction Response:")
for key, val in result.items():
    print(f"  {key}: {val}")

## 3. Direct End-to-End Video Recognition (Video -> MediaPipe -> Model)

In [ ]:
# Example of running inference directly on a video file (.mp4, .avi, .mov):
# video_file = "path/to/user_sign_video.mp4"
# if Path(video_file).exists():
#     video_result = predict_video(video_file, language="ar", predictor=predictor)
#     print("Video Recognition Result:", video_result)
print("Video inference pipeline is ready to accept .mp4, .avi, and .mov files directly via predict_video(video_path)!")

## 4. Vocabulary Inspection (ADR-001 Verification)

In [ ]:
from ml.sign_language.src.preprocessing import SignVocabulary

asl_vocab = SignVocabulary.load(repo_root / "ml/sign_language/artifacts/how2sign/vocab.json")
arsl_vocab = SignVocabulary.load(repo_root / "ml/sign_language/artifacts/arabsign/vocab.json")

print(f"ArabSign (ArSL) Vocab: {len(arsl_vocab)} tokens (Sample: {list(arsl_vocab.token2id.keys())[5:15]})")
print(f"How2Sign (ASL) Vocab: {len(asl_vocab)} tokens (Sample: {list(asl_vocab.token2id.keys())[5:15]})")

# Verify zero vocabulary overlap between languages
overlap = set(arsl_vocab.token2id.keys()) & set(asl_vocab.token2id.keys()) - {"<blank>", "<pad>", "<unk>", "<sos>", "<eos>"}
print(f"Cross-language vocabulary leakage: {len(overlap)} tokens (Zero contamination guaranteed)")